In [24]:
from pyexpat import features
import  os
from keras.src.utils.module_utils import tensorflow
os.path.expanduser('~/.keras/models')
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout,Flatten
import pickle
from tensorflow.keras.applications.vgg16 import VGG16
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [25]:
model_cvv=VGG16(weights="imagenet",input_shape=(150,150,3), include_top=False)


In [26]:
model_cvv.summary()


Model: "vgg16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 150, 150, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 150, 150, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 75, 75, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 75, 75, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 75, 75, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 37, 37, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 37, 37, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 37, 37, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 37, 37, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 18, 18, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 18, 18, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 18, 18, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 18, 18, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 9, 9, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 9, 9, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 9, 9, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 9, 9, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 4, 4, 512)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,714,688 (56.13 MB)

 Trainable params: 14,714,688 (56.13 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:
data_generatorr=ImageDataGenerator(rescale=1.0/255,)
generate=data_generatorr.flow_from_directory('train_transform/',target_size=(150,150),class_mode='binary',batch_size=20)


Found 2000 images belonging to 2 classes.


In [28]:
data_generatorr_test = ImageDataGenerator(rescale=1.0 / 255, )
generate_test = data_generatorr_test.flow_from_directory('test_transform/', target_size=(150, 150), class_mode='binary',batch_size=10)

Found 1400 images belonging to 2 classes.


In [29]:
model_cvv.trainable =True
set_trainable=False
for layer in model_cvv.layers:
    if layer.name=='block5_conv1':
        set_trainable=True
    if set_trainable==True:
        layer.trainable=True
    else:
        layer.trainable=False

In [30]:
model=Sequential()
model.add(model_cvv)
model.add(Flatten())
model.add(Dense(256,activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1,activation='sigmoid'))
model.compile(optimizer='Adam',loss='binary_crossentropy',metrics=['accuracy'])

In [31]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 4, 4, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │     2,097,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,812,353 (64.13 MB)

 Trainable params: 9,177,089 (35.01 MB)

 Non-trainable params: 7,635,264 (29.13 MB)

In [32]:
model.fit(generate,steps_per_epoch=100,epochs=30,validation_data=generate_test,validation_steps=30)

Epoch 1/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 131s 1s/step - accuracy: 0.4865 - loss: 0.8485 - val_accuracy: 0.5367 - val_loss: 0.6929
Epoch 2/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 90s 906ms/step - accuracy: 0.5005 - loss: 0.6932 - val_accuracy: 0.5100 - val_loss: 0.6931
Epoch 3/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 384s 4s/step - accuracy: 0.4920 - loss: 0.6932 - val_accuracy: 0.5033 - val_loss: 0.6931
Epoch 4/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 338s 3s/step - accuracy: 0.4900 - loss: 0.6933 - val_accuracy: 0.5033 - val_loss: 0.6931
Epoch 5/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 91s 910ms/step - accuracy: 0.4910 - loss: 0.6932 - val_accuracy: 0.4933 - val_loss: 0.6932
Epoch 6/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 86s 864ms/step - accuracy: 0.4930 - loss: 0.6932 - val_accuracy: 0.4800 - val_loss: 0.6932
Epoch 7/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 80s 806ms/step - accuracy: 0.4850 - loss: 0.6932 - val_accuracy: 0.5033 - val_loss: 0.6931
Epoch 8/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 127s 1s/step - accuracy: 0.5045 - loss: 0.6932 - 